[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/02_Vision_Language_Models/01_clip_from_scratch.ipynb)

# 01. CLIP from Scratch

**CLIP (Contrastive Language-Image Pre-training)** is the foundation of modern multimodal AI.

**This notebook covers:**
- CLIP architecture — built piece by piece
- Contrastive loss (InfoNCE) — visualized step by step
- Training CLIP on synthetic data (CPU-friendly)
- Visualizing the learned embedding space

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/02_Vision_Language_Models")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from utils.visualization import *
from utils.helpers import *

set_style()
device = get_device()

## 1. CLIP Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('CLIP Architecture', fontsize=20, fontweight='bold', pad=20)

# Image side
draw_architecture_block(ax, 3, 9, 3, 0.7, 'Image', '#E74C3C')
draw_architecture_block(ax, 3, 7.5, 3, 0.9, 'Image Encoder\n(ViT or ResNet)', '#E74C3C')
draw_architecture_block(ax, 3, 5.8, 3, 0.7, 'Image Embedding\n[B, d_img]', '#C0392B')
draw_architecture_block(ax, 3, 4.3, 3, 0.7, 'Projection\nLinear(d_img, d)', '#C0392B')

# Text side
draw_architecture_block(ax, 11, 9, 3, 0.7, 'Text', '#3498DB')
draw_architecture_block(ax, 11, 7.5, 3, 0.9, 'Text Encoder\n(Transformer)', '#3498DB')
draw_architecture_block(ax, 11, 5.8, 3, 0.7, 'Text Embedding\n[B, d_txt]', '#2980B9')
draw_architecture_block(ax, 11, 4.3, 3, 0.7, 'Projection\nLinear(d_txt, d)', '#2980B9')

# L2 normalize
draw_architecture_block(ax, 3, 3.0, 2.5, 0.5, 'L2 Normalize', '#F39C12')
draw_architecture_block(ax, 11, 3.0, 2.5, 0.5, 'L2 Normalize', '#F39C12')

# Similarity matrix
draw_architecture_block(ax, 7, 1.5, 6, 1.2, 'Cosine Similarity Matrix / Temperature\n+ Contrastive Loss (InfoNCE)', '#9B59B6', fontsize=11)

# Arrows
for y_start, y_end in [(8.6, 8.0), (7.0, 6.2), (5.4, 4.7), (3.9, 3.3)]:
    draw_arrow(ax, (3, y_start), (3, y_end))
    draw_arrow(ax, (11, y_start), (11, y_end))

draw_arrow(ax, (3, 2.7), (5, 2.0))
draw_arrow(ax, (11, 2.7), (9, 2.0))

plt.tight_layout()
plt.savefig('../assets/clip_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Build CLIP Step by Step

In [ ]:
class CLIPImageEncoder(nn.Module):
    """Small ViT-style image encoder for CLIP."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 embed_dim=128, n_heads=4, n_layers=3):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        
        self.patch_embed = nn.Conv2d(in_channels, embed_dim, patch_size, patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)  # [B, N, D]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.transformer(x)
        return self.norm(x[:, 0])  # [CLS] output


class CLIPTextEncoder(nn.Module):
    """Small transformer text encoder for CLIP."""
    def __init__(self, vocab_size=5000, embed_dim=128, max_len=32,
                 n_heads=4, n_layers=3):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(x) + self.pos_embed(pos)
        x = self.transformer(x)
        return self.norm(x[:, 0])  # [CLS] output


class CLIP(nn.Module):
    """Full CLIP model with contrastive learning."""
    def __init__(self, embed_dim=128, projection_dim=64):
        super().__init__()
        self.image_encoder = CLIPImageEncoder(embed_dim=embed_dim)
        self.text_encoder = CLIPTextEncoder(embed_dim=embed_dim)
        
        # Projection heads (map to shared space)
        self.image_proj = nn.Linear(embed_dim, projection_dim)
        self.text_proj = nn.Linear(embed_dim, projection_dim)
        
        # Learnable temperature
        self.temperature = nn.Parameter(torch.ones(1) * np.log(1 / 0.07))

    def encode_image(self, images):
        features = self.image_encoder(images)
        projected = self.image_proj(features)
        return F.normalize(projected, dim=-1)

    def encode_text(self, text_ids):
        features = self.text_encoder(text_ids)
        projected = self.text_proj(features)
        return F.normalize(projected, dim=-1)

    def forward(self, images, text_ids):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(text_ids)
        
        # Cosine similarity scaled by temperature
        logit_scale = self.temperature.exp()
        logits_per_image = logit_scale * img_emb @ txt_emb.T
        logits_per_text = logits_per_image.T
        
        return logits_per_image, logits_per_text, img_emb, txt_emb


model = CLIP(embed_dim=128, projection_dim=64)
count_parameters(model)

## 3. The Contrastive Loss (InfoNCE) — Visualized

**Key idea:** In a batch of N image-text pairs:
- The N diagonal entries are **positive pairs** (matching)
- The N²-N off-diagonal entries are **negative pairs** (non-matching)
- Loss = push positives together, push negatives apart

In [ ]:
def clip_loss(logits_per_image, logits_per_text):
    """Symmetric contrastive loss (InfoNCE)."""
    batch_size = logits_per_image.shape[0]
    labels = torch.arange(batch_size, device=logits_per_image.device)
    
    # Image-to-text: which text matches this image?
    loss_i2t = F.cross_entropy(logits_per_image, labels)
    # Text-to-image: which image matches this text?
    loss_t2i = F.cross_entropy(logits_per_text, labels)
    
    return (loss_i2t + loss_t2i) / 2


# Visualize the loss computation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('CLIP Contrastive Loss — Step by Step', fontsize=16, fontweight='bold')

B = 5
img_emb = F.normalize(torch.randn(B, 64), dim=-1)
txt_emb = F.normalize(torch.randn(B, 64), dim=-1)
sim = (img_emb @ txt_emb.T).detach().numpy()

# Step 1: Raw similarity
ax = axes[0]
im = ax.imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Step 1: Cosine Similarity')
ax.set_xlabel('Text')
ax.set_ylabel('Image')
for i in range(B):
    for j in range(B):
        color = 'white' if abs(sim[i,j]) > 0.5 else 'black'
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

# Step 2: Labels (diagonal = positive)
ax = axes[1]
target = np.eye(B)
ax.imshow(target, cmap='Greens')
ax.set_title('Step 2: Target\n(diagonal = matching pairs)')
ax.set_xlabel('Text')
ax.set_ylabel('Image')
for i in range(B):
    for j in range(B):
        label = '✓ match' if i == j else '✗'
        color = 'white' if i == j else 'gray'
        ax.text(j, i, label, ha='center', va='center', fontsize=9, color=color)

# Step 3: After training (what we want)
ax = axes[2]
ideal = np.eye(B) * 0.95 + (1 - np.eye(B)) * (-0.3) + np.random.randn(B, B) * 0.05
np.fill_diagonal(ideal, np.random.uniform(0.85, 0.95, B))
im = ax.imshow(ideal, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Step 3: After Training\n(diagonal HIGH, rest LOW)')
ax.set_xlabel('Text')
ax.set_ylabel('Image')
for i in range(B):
    for j in range(B):
        color = 'white' if abs(ideal[i,j]) > 0.5 else 'black'
        ax.text(j, i, f'{ideal[i,j]:.2f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('../assets/clip_loss_visual.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Train CLIP on Synthetic Data (CPU-Friendly)

In [ ]:
# Create synthetic dataset
images, texts, labels = create_synthetic_image_text_pairs(n_samples=200, img_size=32, n_classes=5)

# Simple tokenizer (word-level)
all_words = set()
for t in texts:
    all_words.update(t.lower().split())
word2id = {w: i+2 for i, w in enumerate(sorted(all_words))}
word2id['[PAD]'] = 0
word2id['[CLS]'] = 1

def tokenize(text, max_len=16):
    ids = [word2id['[CLS]']] + [word2id.get(w, 0) for w in text.lower().split()]
    ids = ids[:max_len]
    ids += [0] * (max_len - len(ids))
    return torch.tensor(ids)

# Create DataLoader
class CLIPDataset(Dataset):
    def __init__(self, images, texts):
        self.images = images
        self.texts = texts
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        return self.images[idx], tokenize(self.texts[idx])

dataset = CLIPDataset(images, texts)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset: {len(dataset)} image-text pairs")
print(f"Vocab: {len(word2id)} words")
print(f"Sample: '{texts[0]}' → {tokenize(texts[0]).tolist()[:8]}...")

In [ ]:
# Train CLIP!
model = CLIP(embed_dim=128, projection_dim=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

losses = []
n_epochs = 30

for epoch in range(n_epochs):
    epoch_loss = 0
    for images_batch, text_batch in loader:
        images_batch = images_batch.to(device)
        text_batch = text_batch.to(device)
        
        logits_i2t, logits_t2i, _, _ = model(images_batch, text_batch)
        loss = clip_loss(logits_i2t, logits_t2i)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f}")

# Plot training curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, linewidth=2, color='#9B59B6')
ax.set_xlabel('Epoch')
ax.set_ylabel('Contrastive Loss')
ax.set_title('CLIP Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the learned similarity matrix
model.eval()
with torch.no_grad():
    test_imgs = torch.stack(images[:10]).to(device)
    test_txts = torch.stack([tokenize(t) for t in texts[:10]]).to(device)
    
    img_emb = model.encode_image(test_imgs)
    txt_emb = model.encode_text(test_txts)

fig = plot_similarity_matrix(
    img_emb, txt_emb,
    labels_a=[f'img_{i}' for i in range(10)],
    labels_b=[t[:15] for t in texts[:10]],
    title='CLIP Learned Similarity (after training)'
)
plt.savefig('../assets/clip_similarity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Diagonal should be brighter (matching pairs have higher similarity)")

## Key Takeaways

1. **CLIP = Image Encoder + Text Encoder + Contrastive Loss**
2. **InfoNCE loss** pulls matching pairs together, pushes non-matching apart
3. **Temperature** controls how "sharp" the similarity distribution is
4. After training, you can do **zero-shot classification** by comparing image embeddings to text embeddings
5. CLIP's shared space enables many downstream tasks without retraining

---
**Next:** `02_image_captioning.ipynb` - Generate text from images